<a href="https://colab.research.google.com/github/Bokenasubi-T/pomacea-distribution-jp/blob/main/03_grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. Aggregate records to grid cells aligned with WorldClim

- Input: `data/pomacea_jp_clean.csv`
- Output: thinned records at 30 arc-seconds (\~1 km) and 5 arc-minutes (\~10 km)
- Note: The grid follows the WorldClim resolution for later overlay with climate data. The cell size itself has no specific ecological meaning, so results will be compared between the two resolutions.

In [2]:
# Load the cleaned data from Google Drive
from google.colab import drive
import pandas as pd
import numpy  as np

drive.mount("/content/drive")
base = "/content/drive/MyDrive/Colab Notebooks/pomacea"
clean = pd.read_csv(f"{base}/data/pomacea_jp_clean.csv")
print(len(clean), "records")

Mounted at /content/drive
260 records


In [5]:
# Thin records to one per grid cell, at two resolutions aligned with WorldClim
resolutions = {"30s(-1 km)": 1/120, "5min (-10 km)": 1/12}

for name, res in resolutions.items():
  cell_x = np.floor(clean["decimalLongitude"] / res).astype(int)
  cell_y = np.floor(clean["decimalLatitude"] / res).astype(int)
  n_cells = len(set(zip(cell_x, cell_y)))
  print(f"{name}: {len(clean)} records -> {n_cells} cells")


30s(-1 km): 260 records -> 205 cells
5min (-10 km): 260 records -> 158 cells


In [7]:
# Keep one record per grid cell (spatial thinning) and save the results
import os
def thin_to_grid(df, res):
  """Keep one record per grid cell and add the coordinates of the cell center."""
  out = df.copy()
  out["cell_x"] = np.floor(out["decimalLongitude"] / res).astype(int)
  out["cell_y"] = np.floor(out["decimalLatitude"] / res).astype(int)
  out = out.drop_duplicates(subset=["cell_x", "cell_y"]).copy()
  out["cell_lon"] = (out["cell_x"] + 0.5) * res # center of the cell
  out["cell_lat"] = (out["cell_y"]+ 0.5) * res
  return out

resolutions = {"30s": 1/120, "5min": 1/12}
thinned = {}

os.makedirs(f"{base}/data", exist_ok=True)
for name, res in resolutions.items():
  t = thin_to_grid(clean, res)
  thinned[name] = t
  t.to_csv(f"{base}/data/pomacea_jp_thinned_{name}.csv", index=False)
  print(f"{name}: {len(clean)} records -> {len(t)} cells")

30s: 260 records -> 205 cells
5min: 260 records -> 158 cells


In [13]:
# Map of thinned records at two resolutions
import folium

m = folium.Map(location=[36, 138],  zoom_start=5, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="GSI pale map"
).add_to(m)

def add_cells(m, cells, res, name, color, opacity):
    """Draw grid cells as rectangles in their own layer."""
    layer = folium.FeatureGroup(name=f"{name} (n={len(cells)})").add_to(m)
    for _, row in cells.iterrows():
        south, west = row["cell_y"] * res, row["cell_x"] * res
        folium.Rectangle(
            bounds=[[south, west], [south + res, west + res]],
            color=color,
            weight=1,
            fill=True,
            fill_opacity=opacity,
        ).add_to(layer)

# Draw the coarse cells first so that the fine cells are shown on top
add_cells(m, thinned["5min"], 1/12,  "5 arc-min cells (~10 km)", "darkorange", 0.3)
add_cells(m, thinned["30s"],  1/120, "30 arc-sec cells (~1 km)", "darkblue",   0.8)

folium.LayerControl(collapsed=False).add_to(m)

m.save(f"{base}/outputs/pomacea_jp_thinned_map.html")
